# requirement & setup

In [ ]:
import os
import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss
from tqdm.auto import tqdm
import lightgbm as lgb

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
plt.style.use('ggplot')
sns.set_palette("muted")
pd.set_option('display.max_columns', None)

BASE_DIR = r"C:\Users\narak\Downloads\sup_ai\ctr-ad-predict"
train_path = os.path.join(BASE_DIR, "train.csv")
test_path = os.path.join(BASE_DIR, "test.csv")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")

# data go

In [ ]:
display(train_df.head())

In [ ]:
display(test_df.head())

In [ ]:
missing_stats = train_df.isnull().sum()
print(missing_stats[missing_stats > 0])

In [ ]:
filtered_df = train_df[["site_id", "app_id", "site_domain", "app_domain"]]
display(filtered_df)

In [ ]:
display(train_df.describe())

In [ ]:
display(train_df['hour'].value_counts().sort_index())

In [ ]:
C_df = train_df[["C1", "C15", "C16", "C21"]].value_counts().reset_index(name='count').sort_values('count', ascending=False).head(10)
display(C_df)

??? Mark **

### Class Imbalance

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='clicked', data=train_df)
plt.title('Distribution of Target (Clicked vs Non-Clicked)')

total = len(train_df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.2f}%'
    x = p.get_x() + p.get_width() / 2 - 0.1
    y = p.get_height() + 1000
    ax.annotate(percentage, (x, y), size=12)

plt.show()

### Time Analysis

In [ ]:
temp_time = pd.to_datetime(train_df['timestamp'], errors='coerce')
train_df['hour_of_day'] = temp_time.dt.hour

plt.figure(figsize=(12, 5))
sns.barplot(x='hour_of_day', y='clicked', data=train_df, ci=None, color='royalblue')
plt.title('Average CTR by Hour of Day')
plt.ylabel('Click Through Rate (CTR)')
plt.xlabel('Hour')
plt.axhline(train_df['clicked'].mean(), color='red', linestyle='--', label='Global Average CTR')
plt.legend()
plt.show()

train_df.drop(columns=['hour_of_day'], inplace=True)

In [ ]:
display(temp_time.dt.hour.value_counts())

In [ ]:
# แปลงเป็นตัวเลขและเรียงลำดับ
train_df['timestamp'] = pd.to_numeric(train_df['timestamp'], errors='coerce')
test_df['timestamp'] = pd.to_numeric(test_df['timestamp'], errors='coerce')

train_df = train_df.sort_values('timestamp').reset_index(drop=True)
test_df = test_df.sort_values('timestamp').reset_index(drop=True)

train_df['is_test'] = 0
test_df['is_test'] = 1
test_df['clicked'] = 0 
df_all = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

### app and site

In [ ]:
# เช็คว่า is_app เป็น True/1 หรือไม่
is_app_mask = df_all['is_app'].astype(str).str.lower().isin(['1', 'true', 'yes'])

# ถ้ายุซเซอร์เข้าผ่านแอป ให้ใช้ app_id ถ้าเข้าผ่านเว็บ ให้ใช้ site_id
df_all['platform_id'] = np.where(is_app_mask, df_all['app_id'], df_all['site_id'])
df_all['platform_domain'] = np.where(is_app_mask, df_all['app_domain'], df_all['site_domain'])

train_tmp = df_all[df_all['is_test'] == 0]
plt.figure(figsize=(8, 4))
sns.barplot(x='is_app', y='clicked', data=train_tmp, ci=None)
plt.title('Average CTR: App vs Site')
plt.show()

### interactive

In [ ]:
# Intent x Relevance ความสนใจ
df_all['user_ad_power_score'] = df_all['historical_user_ctr'] * df_all['ad_quality_score']

# Context x Device สภาพแวดล้อมกับอุปกรณ์
df_all['device_x_banner'] = df_all['device_type'].astype(str) + "_" + df_all['banner_pos'].astype(str)
if 'creative_size' in df_all.columns:
    df_all['device_x_creative'] = df_all['device_type'].astype(str) + "_" + df_all['creative_size'].astype(str)

# User x Context ช่วงเวลาที่คนกลุ่มนี้ชอบเข้า
df_all['user_seg_x_hour'] = df_all['user_segment'].astype(str) + "_" + df_all['hour'].astype(str)

print(['user_ad_power_score', 'device_x_banner', 'device_x_creative', 'user_seg_x_hour'])

In [ ]:
display(df_all[['user_ad_power_score', 'device_x_banner', 'device_x_creative', 'user_seg_x_hour']].head())

In [ ]:
display(df_all["device_x_creative"].value_counts())

In [ ]:
display(df_all["user_seg_x_hour"].value_counts())

### freq encode

In [ ]:
# นับเฉพาะใน Train Set เท่านั้นเผื่อ Leak
train_tmp = df_all[df_all['is_test'] == 0]
freq_map = train_tmp['device_model'].value_counts()

# map ความถี่กลับ
df_all['device_model_freq'] = df_all['device_model'].map(freq_map).fillna(0)

plt.figure(figsize=(8, 4))
sns.histplot(df_all['device_model_freq'], bins=50, kde=True)
plt.title('Distribution of Device Model Frequencies')
plt.show()

In [ ]:
display(df_all["device_model_freq"].value_counts())

### id to past click

In [ ]:
df_train_tmp = df_all[df_all['is_test'] == 0].copy()
df_test_tmp = df_all[df_all['is_test'] == 1].copy()

ctr_targets = [
    ("user_id", "user_ctr"), 
    ("ad_id", "ad_ctr"), 
    ("platform_id", "platform_ctr"),
    (["user_id", "ad_id"], "user_ad_ctr"),
    (["ad_id", "platform_id"], "ad_platform_ctr"),
    (["user_id", "hour"], "user_hour_ctr"), 
    ("hour", "hour_ctr")
]

for cols, new_col in ctr_targets:
    grouped = df_train_tmp.groupby(cols)['clicked']
    df_train_tmp[new_col] = (grouped.cumsum() - df_train_tmp['clicked']) / (grouped.cumcount() + 1)
    
    # Test: ดึงสถิติรวมจาก Train มาใช้
    stats = df_train_tmp.groupby(cols)['clicked'].agg(['sum', 'count']).reset_index()
    df_test_tmp = df_test_tmp.merge(stats, on=cols, how='left')
    df_test_tmp[new_col] = df_test_tmp['sum'].fillna(0) / (df_test_tmp['count'].fillna(0) + 1)
    df_test_tmp.drop(columns=['sum', 'count'], inplace=True)

df_all = pd.concat([df_train_tmp, df_test_tmp], axis=0).reset_index(drop=True)

In [ ]:
df_all[["user_ad_ctr", "ad_platform_ctr", "user_hour_ctr", "hour_ctr"]]

In [ ]:
df_all[["user_ad_ctr", "ad_platform_ctr", "user_hour_ctr", "hour_ctr"]].describe()

# models

In [ ]:
class Config:
    BASE_DIR = r"C:\Users\narak\Downloads\sup_ai\ctr-ad-predict"
    train_path = os.path.join(BASE_DIR, "train.csv")
    test_path = os.path.join(BASE_DIR, "test.csv")
    sample_sub_path = os.path.join(BASE_DIR, "sample_submission.csv")
    output_path = os.path.join(BASE_DIR, "outputs", "submission_feature.csv")
    target = 'clicked'
    
    num_cols = ['ad_quality_score', 'user_depth', 'historical_user_ctr']
    cat_cols = ['hour', 'day_of_week', 'device_type', 'device_model', 'device_conn_type',
                'banner_pos', 'is_app', 'creative_size', 'user_segment']
    
    batch_size = 8192 
    epochs = 40 
    lr = 1e-3
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_workers = 0 
    
    clip_min = 1e-6
    clip_max = 1.0 - 1e-6

In [ ]:
class Preprocessor:
    def __init__(self, rare_threshold=5):
        self.rare_threshold = rare_threshold
        self.vocab_sizes = {}
        self.embed_dims = {}
        self.scaler = StandardScaler()
        self.final_num_cols = Config.num_cols.copy()
        self.final_cat_cols = Config.cat_cols.copy()

    def fit_transform(self, df_train, df_test):
        
        # Leakage Prevention: Time-Based Sorting
        if 'timestamp' in df_train.columns:
            df_train['timestamp'] = pd.to_numeric(df_train['timestamp'], errors='coerce')
            df_test['timestamp'] = pd.to_numeric(df_test['timestamp'], errors='coerce')
            df_train = df_train.sort_values('timestamp').reset_index(drop=True)
            df_test = df_test.sort_values('timestamp').reset_index(drop=True)
            
        df_train['is_test'], df_test['is_test'], df_test[Config.target] = 0, 1, 0
        df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
        
        is_app_mask = df_all['is_app'].astype(str).str.lower().isin(['1', 'true', 'yes'])
        df_all['platform_id'] = np.where(is_app_mask, df_all['app_id'], df_all['site_id'])
        df_all['platform_domain'] = np.where(is_app_mask, df_all['app_domain'], df_all['site_domain'])
        
        # Numerical Interaction: Intent x Relevance
        df_all['user_ad_power_score'] = df_all['historical_user_ctr'] * df_all['ad_quality_score']
        self.final_num_cols.append('user_ad_power_score')
        
        # Categorical Interactions: Context x Device & User x Context
        df_all['device_x_banner'] = df_all['device_type'].astype(str) + "_" + df_all['banner_pos'].astype(str)
        df_all['user_seg_x_hour'] = df_all['user_segment'].astype(str) + "_" + df_all['hour'].astype(str)
        if 'creative_size' in df_all.columns:
            df_all['device_x_creative'] = df_all['device_type'].astype(str) + "_" + df_all['creative_size'].astype(str)
            
        cross_cats = ['platform_id', 'platform_domain', 'device_x_banner', 'user_seg_x_hour', 'device_x_creative']
        for col in cross_cats:
            if col in df_all.columns and col not in self.final_cat_cols: 
                self.final_cat_cols.append(col)

        df_train_tmp = df_all[df_all['is_test'] == 0].copy()
        df_test_tmp = df_all[df_all['is_test'] == 1].copy()
        
        # Frequency Encoding สำหรับ device_model
        freq_map = df_train_tmp['device_model'].value_counts()
        df_train_tmp['device_model_freq'] = df_train_tmp['device_model'].map(freq_map).fillna(0)
        df_test_tmp['device_model_freq'] = df_test_tmp['device_model'].map(freq_map).fillna(0)
        self.final_num_cols.append('device_model_freq')

        # CTR Features (Past-Only)
        ctr_targets = [
            ("user_id", "user_ctr"), ("ad_id", "ad_ctr"), ("platform_id", "platform_ctr"),
            (["user_id", "ad_id"], "user_ad_ctr"), (["ad_id", "platform_id"], "ad_platform_ctr"),
            (["user_id", "hour"], "user_hour_ctr"), ("hour", "hour_ctr")
        ]
        
        for cols, new_col in ctr_targets:
            grouped = df_train_tmp.groupby(cols)[Config.target]
            df_train_tmp[new_col] = (grouped.cumsum() - df_train_tmp[Config.target]) / (grouped.cumcount() + 1)
            
            stats = df_train_tmp.groupby(cols)[Config.target].agg(['sum', 'count']).reset_index()
            df_test_tmp = df_test_tmp.merge(stats, on=cols, how='left')
            df_test_tmp[new_col] = df_test_tmp['sum'].fillna(0) / (df_test_tmp['count'].fillna(0) + 1)
            df_test_tmp = df_test_tmp.drop(columns=['sum', 'count'])
            if new_col not in self.final_num_cols: self.final_num_cols.append(new_col)
            
        df_all = pd.concat([df_train_tmp, df_test_tmp], axis=0).reset_index(drop=True)

        self.final_cat_cols = [c for c in self.final_cat_cols if c in df_all.columns]
        for col in self.final_cat_cols:
            df_all[col] = df_all[col].fillna('MISSING').astype(str)
            counts = df_all[col].value_counts()
            rare_classes = set(counts[counts < self.rare_threshold].index)
            df_all[col] = df_all[col].apply(lambda x: 'OOV' if x in rare_classes else x)
            
            unique_vals = df_all[col].unique()
            val_to_idx = {val: i for i, val in enumerate(unique_vals)}
            df_all[col] = df_all[col].map(val_to_idx)
            
            self.vocab_sizes[col] = len(val_to_idx)
            self.embed_dims[col] = min(50, max(4, int((self.vocab_sizes[col] ** 0.25) * 4)))
            
        self.final_num_cols = [c for c in self.final_num_cols if c in df_all.columns]
        df_all[self.final_num_cols] = self.scaler.fit_transform(df_all[self.final_num_cols].fillna(0))
        
        # drops ส่วนที่ไม่ได้ใช้แล้ว
        drop_cols = ['is_test', 'impression_id', 'timestamp', 'user_id', 'ad_id', 'site_id', 'app_id', 
                     'site_domain', 'app_domain', 'C1', 'C15', 'C16', 'C21']
        df_all = df_all.drop(columns=[c for c in drop_cols if c in df_all.columns])
        return df_all.iloc[:len(df_train_tmp)], df_all.iloc[len(df_train_tmp):]

## data

In [ ]:
class CTRDataset(Dataset):
    def __init__(self, df, feature_names, target_col):
        self.x, self.y = df[feature_names].values.astype(np.float32), df[target_col].values.astype(np.float32)
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]

In [ ]:
def build_dataloader(df, feature_names, target_col, batch_size, is_train=True):
    return DataLoader(CTRDataset(df, feature_names, target_col), batch_size=batch_size, shuffle=is_train, num_workers=Config.num_workers, pin_memory=True)

In [ ]:
class PureFCN(nn.Module):
    def __init__(self, cat_cols, num_cols, vocab_sizes, embed_dims, hidden_units=(256, 128, 64), dropout=0.5):
        super(PureFCN, self).__init__()
        self.cat_cols, self.num_cols, self.num_cat = cat_cols, num_cols, len(cat_cols)
        self.embeddings = nn.ModuleDict({col: nn.Embedding(num_embeddings=vocab_sizes[col], embedding_dim=embed_dims[col]) for col in cat_cols})
        
        input_dim = sum(embed_dims[col] for col in cat_cols) + len(num_cols)
        layers = []
        for hidden_dim in hidden_units:
            layers.extend([nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            input_dim = hidden_dim 
            
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(input_dim, 1)

    def forward(self, x):
        cat_embs = [self.embeddings[col](x[:, i].long()) for i, col in enumerate(self.cat_cols)]
        cat_out = torch.cat(cat_embs, dim=1) if cat_embs else torch.empty(x.size(0), 0, device=x.device)
        return self.output_layer(self.mlp(torch.cat([cat_out, x[:, self.num_cat:]], dim=1)))

# modelssssss

why pure fnc???

I got log using quick validate by myself :DDDDD

Epoch 10/10 | Train Loss: 0.4994 | Val AUC: 0.7526 | Val LogLoss: 0.5327 Deepfm   
Epoch 3/10 | Train Loss: 0.5137 | Val AUC: 0.7656 | Val LogLoss: 0.5166 Deepfm + FCN   
Epoch 4/10 | Train Loss: 0.5134 | Val AUC: 0.7663 | Val LogLoss: 0.5157 Pure FCN   
Epoch 4/10 | Train Loss: 0.5911 | Val AUC: 0.6330 | Val LogLoss: 0.5916 MGNN   

In [ ]:
def train_pure_fcn(model, train_loader, val_loader, epochs, lr, pos_weight):
    model.to(Config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(Config.device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    best_auc, best_weights, patience, epochs_no_improve = 0.0, None, 8, 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        
        for batch_x, batch_y in train_pbar:
            batch_x, batch_y = batch_x.to(Config.device), batch_y.to(Config.device)
            optimizer.zero_grad()
            if torch.cuda.is_available():
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(batch_x), batch_y.unsqueeze(1).float())
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(batch_x), batch_y.unsqueeze(1).float())
                loss.backward()
                optimizer.step()
            train_loss += loss.item()
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                probs = torch.sigmoid(model(batch_x.to(Config.device))).cpu().numpy().flatten()
                probs = np.clip(probs, Config.clip_min, Config.clip_max)
                val_preds.extend(probs) 
                val_targets.extend(batch_y.numpy().flatten())
                
        val_auc, val_logloss = roc_auc_score(val_targets, val_preds), log_loss(val_targets, val_preds)
        print(f"Epoch {epoch+1}/{epochs} | LR: {optimizer.param_groups[0]['lr']:.1e} | Loss: {train_loss/len(train_loader):.4f} | Val AUC: {val_auc:.4f} | Val LogLoss: {val_logloss:.4f}")
        
        scheduler.step(val_auc)
        if val_auc > best_auc:
            best_auc, best_weights, epochs_no_improve = val_auc, copy.deepcopy(model.state_dict()), 0
            print("Best model updated")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping")
                break

    model.load_state_dict(best_weights)
    return model

# data implement again and again

In [ ]:
train_df = pd.read_csv(Config.train_path)
test_df = pd.read_csv(Config.test_path)
sub_df = pd.read_csv(Config.sample_sub_path)

preprocessor = Preprocessor()
train_proc, test_proc = preprocessor.fit_transform(train_df, test_df)

In [ ]:
# Index Slicing
split_idx = int(len(train_proc) * 0.85)
train_set = train_proc.iloc[:split_idx].copy()
val_set = train_proc.iloc[split_idx:].copy()
    
print(f"Train size: {len(train_set):,}, Val size: {len(val_set):,}")

In [ ]:
# Class Imbalance
num_pos = train_set[Config.target].sum()
num_neg = len(train_set) - num_pos
pos_weight_val = np.sqrt(num_neg / num_pos) 
print(f"Soft Class Imbalance Pos Weight: {pos_weight_val:.2f}")

feature_names = preprocessor.final_cat_cols + preprocessor.final_num_cols

train_loader = build_dataloader(train_set, feature_names, Config.target, batch_size=Config.batch_size, is_train=True)
val_loader = build_dataloader(val_set, feature_names, Config.target, batch_size=Config.batch_size, is_train=False)
test_loader = build_dataloader(test_proc, feature_names, Config.target, batch_size=Config.batch_size, is_train=False)

# Main extract

In [ ]:
if __name__ == '__main__':
    print("\nTraining Pure FCN\n")
    fcn_model = PureFCN(preprocessor.final_cat_cols, preprocessor.final_num_cols, preprocessor.vocab_sizes, preprocessor.embed_dims)
    best_fcn = train_pure_fcn(fcn_model, train_loader, val_loader, epochs=Config.epochs, lr=Config.lr, pos_weight=pos_weight_val)
    
    best_fcn.eval()
    fcn_val_preds, fcn_test_preds = [], []
    with torch.no_grad():
        for batch_x, _ in val_loader: 
            fcn_val_preds.extend(np.clip(torch.sigmoid(best_fcn(batch_x.to(Config.device))).cpu().numpy().flatten(), Config.clip_min, Config.clip_max))
        for batch_x, _ in test_loader: 
            fcn_test_preds.extend(np.clip(torch.sigmoid(best_fcn(batch_x.to(Config.device))).cpu().numpy().flatten(), Config.clip_min, Config.clip_max))
    fcn_val_preds, fcn_test_preds = np.array(fcn_val_preds), np.array(fcn_test_preds)

    print("\nTraining LightGBM\n")
    cat_idx = [feature_names.index(c) for c in preprocessor.final_cat_cols if c in feature_names]
    train_data = lgb.Dataset(train_set[feature_names], label=train_set[Config.target], categorical_feature=cat_idx)
    val_data = lgb.Dataset(val_set[feature_names], label=val_set[Config.target], reference=train_data, categorical_feature=cat_idx)
    
    lgb_params = {
        'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.01, 'num_leaves': 31, 'max_depth': 6,
        'min_data_in_leaf': 100, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'scale_pos_weight': pos_weight_val, 'lambda_l1': 0.1, 'lambda_l2': 2.0, 'verbose': -1, 'random_state': 42, 'n_jobs': -1
    }
    lgb_model = lgb.train(lgb_params, train_data, num_boost_round=3000, valid_sets=[train_data, val_data], 
                          callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)])
    
    lgb_val_preds = np.clip(lgb_model.predict(val_set[feature_names]), Config.clip_min, Config.clip_max)
    lgb_test_preds = np.clip(lgb_model.predict(test_proc[feature_names]), Config.clip_min, Config.clip_max)

    print("\nENSEMBLE & ISOTONIC CALIBRATION\n")
    y_val_true = val_set[Config.target].values
    best_weight, best_blend_auc = 0.5, 0.0
    
    for w in np.arange(0.0, 1.05, 0.05):
        blend_auc = roc_auc_score(y_val_true, (w * lgb_val_preds) + ((1 - w) * fcn_val_preds))
        if blend_auc > best_blend_auc: best_blend_auc, best_weight = blend_auc, w
            
    print(f"Optimal Blend -> LGBM: {best_weight:.2f} | FCN: {1 - best_weight:.2f} | MIX AUC: {best_blend_auc:.4f}")
    
    final_val_preds = (best_weight * lgb_val_preds) + ((1 - best_weight) * fcn_val_preds)
    final_test_preds = (best_weight * lgb_test_preds) + ((1 - best_weight) * fcn_test_preds)

    iso_reg = IsotonicRegression(out_of_bounds='clip')
    iso_reg.fit(final_val_preds, y_val_true)
    
    calibrated_val_preds = np.clip(iso_reg.predict(final_val_preds), Config.clip_min, Config.clip_max)
    calibrated_test_preds = np.clip(iso_reg.predict(final_test_preds), Config.clip_min, Config.clip_max)
    
    sub_df[Config.target] = calibrated_test_preds
    os.makedirs(os.path.dirname(Config.output_path), exist_ok=True)
    sub_df.to_csv(Config.output_path, index=False)
    
    print(f"LogLoss AFTER Calibration & Clipping: {log_loss(y_val_true, calibrated_val_preds):.5f}")
    print(f"\nV9 Endgame Sub saved to: {Config.output_path}")